# Fine-tune LLMs for CTF/Coding - Self-Contained Pipeline

Complete pipeline: data scraping → synthesis → training → export

**Setup:**
- Runtime → Change runtime type → **T4 GPU**
- Run cells top-to-bottom (1-9)
- No external files needed - everything is built in

In [ ]:
# Cell 1: Install all dependencies
%%capture
import os, re, sys, json, time, re, shutil, tempfile
from pathlib import Path

if "COLAB_" in "".join(os.environ.keys()):
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer requests gitpython pyyaml tqdm
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
    torch._dynamo.config.recompile_limit = 64

import requests, yaml
from git import Repo
from datasets import load_dataset
from tqdm import tqdm
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration
CONFIG = {
    "model_name": "unsloth/Qwen3.5-4B",
    "max_seq_length": 4096,
    "lora_r": 16,
    "lora_alpha": 16,
    "batch_size": 1,
    "grad_accum": 4,
    "num_epochs": 3,
    "learning_rate": 2e-4,
    "max_per_repo": 200,
    "max_hf_samples": 3000,
    "output_dir": "/content/outputs/qwen4b-ctf",
}
print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 3: Scrape CTF writeups from GitHub
CTF_WRITEUP_REPOS = [
    {"url": "https://github.com/Cajac/picoCTF-Writeups", "name": "picoctf-cajac", "category": "picoctf"},
    {"url": "https://github.com/vivian-dai/PicoCTF2021-Writeup", "name": "picoctf-2021", "category": "picoctf"},
    {"url": "https://github.com/DarkCodeOrg/CryptoHack", "name": "cryptohack-darkcode", "category": "cryptohack"},
    {"url": "https://github.com/AyushSingh-c/Cryptohack", "name": "cryptohack-ayush", "category": "cryptohack"},
    {"url": "https://github.com/H3xKatana/pwncollege-writeups", "name": "pwncollege-h3x", "category": "pwncollege"},
    {"url": "https://github.com/prettyb0iisam/pwncollege-writeups", "name": "pwncollege-prettyb0i", "category": "pwncollege"},
    {"url": "https://github.com/id-none/pwncollege_writeup", "name": "pwncollege-idnone", "category": "pwncollege"},
    {"url": "https://github.com/Adamkadaban/CTFs", "name": "ctfs-adamkadaban", "category": "multi"},
    {"url": "https://github.com/Cryptogenic/Exploit-Writeups", "name": "exploit-writeups", "category": "pwn"},
    {"url": "https://github.com/ffffffff0x/1earn", "name": "0x1earn", "category": "multi"},
]

def clone_repo(url, dest):
    dest_path = Path(dest)
    if dest_path.exists():
        shutil.rmtree(dest_path)
    try:
        Repo.clone_from(url, dest, depth=1)
        return dest_path.exists()
    except Exception as e:
        print(f"  Failed: {e}")
        return False

def find_solution_boundary(content):
    markers = [r'^##\s+(?:Solution|Writeup|Exploit|Answer|Flag)',
               r'^###\s+(?:Solution|Writeup|Exploit|Answer|Flag)',
               r'\*\*(?:Solution|Exploit|Answer|Flag)\*\*']
    lines = content.split('\n')
    for i, line in enumerate(lines):
        for marker in markers:
            if re.match(marker, line, re.IGNORECASE):
                return i
    return len(lines)

def extract_code_blocks(content):
    pattern = r'```(\w+)?\n(.*?)```'
    return [{"lang": lang or "text", "code": code.strip()}
            for lang, code in re.findall(pattern, content, re.DOTALL)
            if len(code.strip()) > 10]

def extract_writeup(md_file, category):
    try:
        content = md_file.read_text(errors='ignore')
        if len(content) < 100:
            return None
        # Skip root READMEs (index pages)
        if md_file.name.lower() == "readme.md":
            return None
        challenge_name = md_file.stem.replace("-", " ").replace("_", " ").title()
        boundary = find_solution_boundary(content)
        description = re.sub(r'```.*?```', '', '\n'.join(content.split('\n')[:boundary]), flags=re.DOTALL)
        description = re.sub(r'\n{3,}', '\n\n', description).strip()[:2000]
        solution = re.sub(r'\n{3,}', '\n\n', '\n'.join(content.split('\n')[boundary:])).strip()[:3000]
        if not description or len(description) < 50:
            return None
        code_blocks = extract_code_blocks(content)
        for cf in list(md_file.parent.glob("*.py"))[:2] + list(md_file.parent.glob("*.c"))[:2]:
            try:
                code = cf.read_text(errors='ignore')
                if len(code) > 10:
                    code_blocks.append({"lang": cf.suffix[1:] or "text", "code": code})
            except: pass
        if code_blocks:
            output = f"## Solution\n\n{solution}\n\n" + "\n\n".join(
                f"```{b['lang']}\n{b['code'][:1500]}\n```" for b in code_blocks[:3])
        else:
            output = f"## Solution\n\n{solution}" or f"## Solution\n\n{description}"
        if len(output) < 50:
            return None
        instruction = "Write an exploit/solution for this challenge" if code_blocks else "Explain how to solve this challenge step by step"
        return {
            "instruction": instruction,
            "input": f"Challenge: {challenge_name}\nCategory: {category}\n\n{description}",
            "output": output
        }
    except: return None

all_writeups = []
with tempfile.TemporaryDirectory() as tmpdir:
    for repo in tqdm(CTF_WRITEUP_REPOS, desc="Repos"):
        dest = f"{tmpdir}/{repo['name']}"
        if clone_repo(repo["url"], dest):
            repo_path = Path(dest)
            count = 0
            for md in list(repo_path.rglob("*.md")) + list(repo_path.rglob("*.MD")):
                if md.name.lower() == "readme.md":
                    continue
                ex = extract_writeup(md, repo["category"])
                if ex:
                    all_writeups.append(ex)
                    count += 1
                    if count >= CONFIG["max_per_repo"]:
                        break
            print(f"  {repo['name']}: {count} examples")

print(f"\n✓ Extracted {len(all_writeups)} writeups")

In [ ]:
# Cell 4: Download HuggingFace datasets
HF_DATASETS = [
    ("Jacqkues/ctf_webserver_v0.1", 340),
    ("kyleavery/picoctf", 120),
    ("justinwangx/CTFtime", CONFIG["max_hf_samples"]),
    ("nvidia/OpenCodeReasoning", CONFIG["max_hf_samples"]),
    ("AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1", CONFIG["max_hf_samples"]),
    ("ayshajavd/code-security-vulnerability-dataset", CONFIG["max_hf_samples"]),
]

hf_examples = []
for name, max_samples in tqdm(HF_DATASETS, desc="HF datasets"):
    try:
        for split in ["train", "test"]:
            try:
                ds = load_dataset(name, split=split)
                break
            except: continue
        else:
            ds = load_dataset(name)
        count = 0
        for item in tqdm(ds, desc=f"  {name.split('/')[-1]}", leave=False):
            if count >= max_samples:
                break
            q = item.get("question", item.get("problem", item.get("description", "")))
            a = item.get("answer", item.get("solution", item.get("flag", "")))
            if item.get("messages"):
                msgs = item["messages"]
                if len(msgs) >= 2:
                    q = msgs[0].get("content", "")
                    a = msgs[-1].get("content", "")
            text = item.get("text_chunk", "")
            if text and not q:
                q = "Explain this CTF writeup and provide the solution"
                a = text
            if q and a:
                hf_examples.append({"instruction": q, "input": "", "output": a})
                count += 1
        print(f"  {name}: {count} examples")
    except Exception as e:
        print(f"  Failed {name}: {e}")

print(f"\n✓ Downloaded {len(hf_examples)} HF examples")

In [ ]:
# Cell 5: Scrape security documentation
DOC_SOURCES = [
    {"url": "https://raw.githubusercontent.com/Gallopsled/pwntools-write-ups/master/README.md", "name": "pwntools"},
    {"url": "https://raw.githubusercontent.com/angr/angr/master/README.md", "name": "angr"},
    {"url": "https://raw.githubusercontent.com/Z3Prover/z3/master/README.md", "name": "z3"},
    {"url": "https://raw.githubusercontent.com/pwndbg/pwndbg/dev/FEATURES.md", "name": "pwndbg"},
    {"url": "https://raw.githubusercontent.com/JonathanSalwan/ROPgadget/master/README.md", "name": "ropgadget"},
    {"url": "https://raw.githubusercontent.com/sashs/Ropper/master/README.md", "name": "ropper"},
    {"url": "https://raw.githubusercontent.com/david942j/one_gadget/master/README.md", "name": "one_gadget"},
    {"url": "https://raw.githubusercontent.com/sympy/sympy/master/README.rst", "name": "sympy"},
    {"url": "https://raw.githubusercontent.com/gmpy2/gmpy2/master/README.md", "name": "gmpy2"},
    {"url": "https://raw.githubusercontent.com/sagemath/sage/master/README.md", "name": "sagemath"},
]

doc_examples = []
for src in tqdm(DOC_SOURCES, desc="Docs"):
    try:
        r = requests.get(src["url"], timeout=30)
        r.raise_for_status()
        content = r.text
        if len(content) < 100:
            continue
        sections = re.split(r'\n(?=## )', content)
        for sec in sections[:20]:
            if len(sec) < 50:
                continue
            title = re.match(r'##\s+(.+)', sec)
            title = title.group(1) if title else "Documentation"
            doc_examples.append({
                "instruction": f"Explain how to use {src['name']} for: {title}",
                "input": sec[:1500],
                "output": f"Based on {src['name']} docs:\n\n{sec[:1500]}"
            })
        print(f"  {src['name']}: {len(sections[:20])} sections")
    except Exception as e:
        print(f"  Failed {src['name']}: {e}")

print(f"\n✓ Scraped {len(doc_examples)} doc examples")

In [ ]:
# Cell 6: Convert to ChatML and merge
SYSTEM_CTF = """You are an expert CTF player. You specialize in:
- Binary exploitation: buffer overflows, ROP, heap, format strings
- Reverse engineering: binary analysis, deobfuscation
- Web: SQL injection, XSS, SSRF, deserialization
- Cryptography: cryptanalysis, key recovery
- Forensics: memory/network analysis, steganography

Analyze systematically, identify the vulnerability, provide step-by-step solution with code."""

SYSTEM_CODING = """You are an expert competitive programmer. You specialize in:
- Algorithm design and analysis
- Data structure optimization
- Code optimization and debugging
- Security-aware coding

Understand requirements, identify optimal approach, write clean efficient code."""

def is_ctf(text):
    return any(k in text.lower() for k in ["pwn", "rev", "web", "crypto", "ctf", "exploit", "vuln"])

all_raw = all_writeups + hf_examples + doc_examples
print(f"Total raw examples: {len(all_raw)}")

chatml_data = []
for ex in tqdm(all_raw, desc="Converting to ChatML"):
    instr = ex.get("instruction", "")
    inp = ex.get("input", "")
    out = ex.get("output", "")
    if not instr or not out:
        continue
    user_content = f"{instr}\n\n{inp}" if inp else instr
    sys_prompt = SYSTEM_CTF if is_ctf(instr) or is_ctf(out) else SYSTEM_CODING
    chatml_data.append({
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_content[:3000]},
            {"role": "assistant", "content": out[:3000]}
        ]
    })

print(f"\n✓ Generated {len(chatml_data)} ChatML examples")

# Save merged file
os.makedirs("/content/data/merged", exist_ok=True)
with open("/content/data/merged/train.jsonl", "w") as f:
    for ex in chatml_data:
        f.write(json.dumps(ex) + "\n")
print(f"✓ Saved to /content/data/merged/train.jsonl")

In [ ]:
# Cell 7: Load model + apply chat template + configure LoRA
from unsloth import FastLanguageModel, get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=True,
    dtype=None,
)
print(f"✓ Model loaded: {CONFIG['model_name']}")
print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=CONFIG["max_seq_length"],
)
print(f"✓ LoRA configured (r={CONFIG['lora_r']})")

from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": "/content/data/merged/train.jsonl"}, split="train")
print(f"✓ Dataset: {len(dataset)} examples")

def fmt(examples):
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in examples["messages"]]}

dataset = dataset.map(fmt, batched=True)
print("✓ Chat template applied")

In [ ]:
# Cell 8: Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_seq_length=CONFIG["max_seq_length"],
        per_device_train_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        warmup_steps=10,
        num_train_epochs=CONFIG["num_epochs"],
        learning_rate=float(CONFIG["learning_rate"]),
        logging_steps=10,
        output_dir="/content/outputs",
        optim="adamw_8bit",
        seed=3407,
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataset_text_field="text",
        dataset_num_proc=1,
        report_to="none",
    ),
)

print("Starting training...")
print(f"  Epochs: {CONFIG['num_epochs']}, Batch: {CONFIG['batch_size']}, Grad accum: {CONFIG['grad_accum']}")
trainer.train()
print("\n✓ Training complete!")

In [ ]:
# Cell 9: Save and download
from google.colab import files

OUTPUT_DIR = CONFIG["output_dir"]
os.makedirs(f"{OUTPUT_DIR}/lora", exist_ok=True)

model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print(f"✓ LoRA saved")

model.save_pretrained_gguf(f"{OUTPUT_DIR}/gguf", tokenizer, quantization_method="q4_k_m")
print(f"✓ GGUF saved")

model.save_pretrained_merged(f"{OUTPUT_DIR}/merged", tokenizer, save_method="merged_16bit")
print(f"✓ Merged model saved")

shutil.make_archive("/content/qwen4b-ctf-output", "zip", "/content/outputs")
files.download("/content/qwen4b-ctf-output.zip")
print("\n✓ Download started!")

In [ ]:
# Cell 10: (Optional) Test the model
from transformers import TextStreamer

messages = [
    {"role": "system", "content": SYSTEM_CTF},
    {"role": "user", "content": "Explain how a buffer overflow exploit works."},
    ]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
)
print("\n✓ Test complete")